# STEP 00 보충. 보조금24 전처리 재검증

## 목적과 선행 단계
보조금24 내부 원문·파생 데이터의 정합성과 신청경로·자격 경계 사례를 검토한다. `00_data_preparation.ipynb`의 보조금24 품질 점검을 보완하며 SSIS↔보조금24 동일사업 여부를 확정하지 않는다.

## 입력
| 경로 | 기준·역할 |
|---|---|
| `data/raw/gov24/gov24_{serviceList,serviceDetail,supportConditions}.csv` | 2026-10-31 수집, 서비스 10,942건, 불변 원문 |
| `data/raw/gov24/_collection.json` | 수집 당시 건수·해시·시점 |
| `data/processed/gov24_{merged,eligibility}.csv` | 기존 생성물, 읽기 전용 대조 기준 |
| `logs/validation/gov24_eligibility_review_2026-10-31.json` | 이전 검토 표본 ID 제외 기준 |
| `logs/validation/gov24_*ai_review_2026-11-05.json` | 이 작업에서 원문을 읽고 기록한 AI 판정, 사람 정답 아님 |
| `logs/validation/gov24_q2_reference_2026-11-05.json` | Q2 브랜치의 메타데이터 읽기 전용 스냅샷 |

## 실제 출력
| 유형 | 경로 | 내용·상태 |
|---|---|---|
| 데이터 | `data/processed/gov24_preparation_v2/gov24_merged_v2.csv` | 전체 상세 필드 무손실 보존, 저장 후 재검증 |
| 데이터 | `data/processed/gov24_preparation_v2/gov24_eligibility_v2.csv` | 수정 규칙 + 별도 AI 후보·근거·사유 |
| 표 | `outputs/00_gov24_preparation/tables/*.csv` | 원본 해시·상세 필드·코드 차이·방문 회귀·AI 검토·건수·눈가림 검수 시트 |
| HTML | `outputs/00_gov24_preparation/summary/00_gov24_preparation_summary_final.html` | 단일 파일 요약, 정적 검증·화면 미검증 |
| 로그 | `logs/validation/gov24_preparation_2026-11-05.json` | 입력·코드·출력 해시, 보호 파일, 한계 |
| 그림 | 해당 없음 | 필드·버전·사례를 표로 보고, 불필요한 그림을 생성하지 않음 |

## 구성
환경 → 결정 기록 → 재실행 → 입력·상세 원문 검증 → 코드·CSV 대조 → AI 원문 검토 → 산출물 정적 검증 → 인수 기록.

In [1]:
from pathlib import Path
import sys, json, hashlib, platform, subprocess
import pandas as pd
ROOT = Path.cwd() if (Path.cwd()/"src").exists() else Path.cwd().parent
assert (ROOT/"src/preprocess/gov24_preparation.py").is_file()
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.preprocess.gov24_preparation import run, STAGE, sha, read
from IPython.display import display, Markdown
print({"python":platform.python_version(),"pandas":pd.__version__,"stage":STAGE})

{'python': '3.13.9', 'pandas': '2.3.3', 'stage': '00_gov24_preparation'}


### 실행 환경 예외

이 환경에서 Jupyter 커널 실행 시 연결이 완료되지 않아 중단했다. 대체로 새 Python 프로세스의 빈 네임스페이스에서 코드 셀을 위에서 아래로 실행하고 실제 표·표준출력을 노트북에 저장했다. 이전 노트북 상태를 사용하지 않았다. 명령줄 재현도 별도로 성공했으나, 일반 Jupyter 커널 시작·실행 검증은 완료로 주장하지 않는다. 이 예외는 이 보충 노트북 실행에만 적용한다.

## Decision Box · 실행·보존·해석 원칙

- 기존 생성물과 Q2/SSIS 파일은 갱신하지 않는다. v2 전용 디렉터리를 사용한다. 아래 `refresh=True`는 이 보충 단계가 만든 표·v2·요약만 갱신하며, 사람 검수 기입 시트는 재실행으로 덮어쓰지 않는다.
- 목록·상세의 동명 필드는 원문이 다를 수 있으므로 상세를 `detail_`로 별도 보존한다. CSV는 UTF-8 BOM·문자열 로딩이며 빈칸을 결측으로 사용한다. 실제 `NA` 문자열은 보존한다.
- 기존 코드 기준은 커밋 `0d86a1d`의 소스로 고정한다. 기존 CSV와 기준 코드, 같은 입력에서 기준/수정 코드, 기존 CSV와 현재 원문 기반 v2를 각각 대조한다. 데이터 줄바꿈 차이와 규칙 변경 효과를 섞지 않는다.
- 웹사이트 다음 줄의 실제 방문신청이 삭제되는 3건을 회귀 검증한다. ‘별도 신청’만으로 신청 불필요를 부여하지 않으며, 자동신청은 선행 가입 조건과 함께 읽는다.
- AI가 미확인 163건과 신규 경계 16건을 실제 원문으로 검토했다. AI 후보는 규칙의 `apply_channel`을 덮어쓰지 않는다. 사람 정답·2인 독립 판정·성능 평가로 표현하지 않는다.
- 연령·소득의 최대·최소 하나로 개인 자격을 판정하지 않는다. 자녀 나이, 복수 대상군, 일부 선발인원 예외와 사업별 기준을 분리한다.
- Q2 후보 2,215건은 확정 동일사업이 아니다. Q2의 128열 마스터와 현재 다른 작업의 마스터를 교체하지 않는다.

In [2]:
manifest = run(refresh=True)
TBL=ROOT/"outputs"/STAGE/"tables"
display(read(TBL/"preparation_counts.csv"))
print(manifest["overall_status"])

,metric,n
0,all_services,10942
1,personal_services,9488
2,youth_overlap_provisional,7824
3,legacy_personal_channel_unknown,163
4,v2_rule_personal_channel_unknown,161
5,ai_reviewed_legacy_unknown,163
6,ai_channel_candidates,38
7,ai_channel_still_unspecified,125
8,ai_new_boundary_review,16
9,independent_human_judgments,0


부분 완료: 기계적 검증·AI 검토 완료, 역사 원본·사람 독립 검증 미완료


## PART A. 수집 기록과 상세 원문 보존

현재 목록과 기존 병합본의 22열 정규화 값을 대조한다. 11월 2일 목록 원 바이트 해시 재구성 성공 기록은 보존했으나, 현재 기존 병합본도 내부 CRLF가 LF로 정규화되어 그 성공을 다시 재현하지 못했다. 상세 전 20개 비키 필드의 현재 원본→v2 저장 일치를 확인한다. 기존 병합본에 들어간 7개 상세 필드만 정규화 값 대조가 가능하며, 남은 13개 필드의 수집 당시 완전 일치를 주장하지 않는다.

In [3]:
display(read(TBL/"raw_integrity.csv"))
detail_check=read(TBL/"detail_field_validation.csv")
display(detail_check)
assert manifest["list_normalized_changed_cells"]==0
print({"previous_list_hash_reconstruction_reproduced":manifest["list_reconstruction_matches_collection_sha256"],"legacy_merged_CR_bytes":manifest["legacy_merged_CR_bytes"]})
assert detail_check.current_raw_to_v2_changed_cells.eq("0").all()
assert detail_check.loc[detail_check.historical_reference.eq("legacy merged"),"historical_normalized_changed_cells"].astype(float).eq(0).all()
assert manifest["protected_unchanged"]
print("원본·기존 파생물·SSIS·로컬 Q2 보호 파일 해시 변화 없음")

,file,rows,columns,bytes,sha256,collection_sha256,collection_hash_match,collection_bytes,CR_bytes
0,gov24_serviceList.csv,10942,22,12563633,10e323344eba8bfbf0127fe5e032711bb4e5ec87d6e287...,0f4961d48cb7e4829c33b34f5490e4e38f81fb48158bdc...,False,12633115,0
1,gov24_serviceDetail.csv,10942,21,14450191,7f0ac4fe0f34571309400231844ef4902404f8e3bdc246...,5f020d6dd4a772422fa6e9d16e1ca11db9f50a47b65a3f...,False,14559830,0
2,gov24_supportConditions.csv,10942,50,1385863,c9b974ce0399fba050cd41f896b6dfe1a5a890a04e7c82...,c9b974ce0399fba050cd41f896b6dfe1a5a890a04e7c82...,True,1385863,0


,field,current_raw_to_v2_changed_cells,historical_reference,historical_normalized_changed_cells,missing_rows
0,공무원확인구비서류,0,원수집 스냅샷 미보관,,0
1,구비서류,0,legacy merged,0.0,0
2,문의처,0,legacy merged,0.0,0
3,법령,0,legacy merged,0.0,5702
4,본인확인필요구비서류,0,원수집 스냅샷 미보관,,0
5,서비스명,0,원수집 스냅샷 미보관,,0
6,서비스목적,0,legacy merged,0.0,5349
7,선정기준,0,원수집 스냅샷 미보관,,9896
8,소관기관명,0,원수집 스냅샷 미보관,,0
9,수정일시,0,원수집 스냅샷 미보관,,0


{'previous_list_hash_reconstruction_reproduced': False, 'legacy_merged_CR_bytes': 0}
원본·기존 파생물·SSIS·로컬 Q2 보호 파일 해시 변화 없음


## PART B. 코드–CSV 버전 차이와 방문 회귀

아래는 차이가 발생한 열만 표시한다. 전체 열별 건수와 변경 셀의 ID·전후 값은 CSV에 보존한다. 기존 CSV의 방문 텍스트 플래그 3건은 True이지만 기존 커밋 코드로 다시 만들면 False가 된다. 수정 후 True로 회복하며, 이 3건의 최종 신청 채널은 목록 범주 때문에 바뀌지 않는다. 근거 문장의 차이는 실제 조건 변경 건수와 다르다.

In [4]:
diff=read(TBL/"code_csv_difference_counts.csv")
display(diff[diff.changed_rows.astype(int).gt(0)])
visit=read(TBL/"visit_regex_regression.csv")
display(visit)
assert len(visit)==3 and visit.fixed_visit_text.eq("True").all()
assert visit.final_channel_changed.eq("False").all()

,comparison,column,changed_rows
12,stored_vs_baseline_code,elig_age_evidence,6
29,stored_vs_baseline_code,apply_visit_text,3
38,stored_vs_baseline_code,apply_evidence,4526
76,baseline_vs_fixed_same_inputs,apply_visit_text,3
81,baseline_vs_fixed_same_inputs,apply_not_needed,5
83,baseline_vs_fixed_same_inputs,apply_channel,5
84,baseline_vs_fixed_same_inputs,apply_channel_src,35
85,baseline_vs_fixed_same_inputs,apply_evidence,814
93,baseline_vs_fixed_same_inputs,info_known_count,3
106,stored_vs_v2_current_raw,elig_age_evidence,6


,서비스ID,서비스명,method_raw,baseline_visit_text,fixed_visit_text,stored_visit_text,final_channel_changed
0,138300000066,양육비 이행 지원 서비스,온라인신청 : https://www.childsupport.or.kr\n방문신청 :...,False,True,True,False
1,314000000272,양천형 밤샘 긴급돌봄 어린이집 운영,온라인: 양천구청 홈페이지\n방문: 어린이집\n유선전화 : 양천구청 당직실(2620...,False,True,True,False
2,O00105200003,팔복예술공장 대관 감면,온라인신청 (https://palbokart.kr/main/inner.php?sMe...,False,True,True,False


## PART C. AI가 수행한 원문 검토

163건 전수에서 38건의 추가 신청·현장처리 방식 후보를 기록하고 125건은 미확인으로 유지했다. 38건에는 온라인 12, 기관 경유 10, 전화·우편·이메일 범주 8, 현장 기기·이용 시 처리 5, 조건부 자동신청 2, 문자 1건이 있다. 원문에서 경로가 확인돼도 현재 신청 가능·완전 비대면을 뜻하지 않는다.

추가 경계 16건은 기존 검토 ID와 겹치지 않게 연령 차이 8건, 소득 경계/불일치 8건을 난수 20261105로 추출했다. 위험 기반 표본이라 전체 오류율을 추정하지 않는다. 원문을 읽고 자녀·가구원 나이, 복수 사업, 구매/발급 단계, 10% 한정 예외 등을 구분했다. 이 AI 검토는 독립 2인 사람 검수의 대체 증빙이 아니다.

In [5]:
ai=read(TBL/"unknown_channel_ai_review.csv")
display(ai.groupby(["ai_code","ai_channel_candidate"]).size().rename("n").reset_index())
boundary=read(TBL/"boundary_ai_review.csv")
display(boundary[["서비스ID","item","ai_issue_type","evidence","ai_judgment"]])
assert len(ai)==163 and ai.ai_channel_candidate.ne("명시 없음").sum()==38
assert len(boundary)==16 and ai.human_verified.eq("False").all()
display(Markdown("**한계:** 현재 규칙 기준 개인 미확인 건수는 기존 163건에서 조건부 자동신청 2건이 해소되어 161건이다. 별도로 신청불필요 오분류 1건이 미확인으로 바뀌었으나 비개인 대상이라 이 분모에 포함되지 않는다. AI 38건은 기존 163건에 대한 별도 검토 수이며 규칙 미확인 건수에서 일괄 차감하지 않는다."))

,ai_code,ai_channel_candidate,n
0,AUTOMATIC_CONDITIONAL,신청 불필요,2
1,CHANNEL_UNSPECIFIED,명시 없음,77
2,CLOSED,명시 없음,4
3,EMAIL,전화·우편·이메일,3
4,FAX,전화·우편·이메일,1
5,INQUIRY_ONLY,명시 없음,24
6,INSTITUTION,기관 경유(본인 신청 아님),10
7,LINKED_PROCEDURE,명시 없음,3
8,ONLINE,온라인,12
9,ON_SITE,현장 기기/이용 시 처리,5


,서비스ID,item,ai_issue_type,evidence,ai_judgment
0,GMW000000210,age,multiple_benefit_groups,필수 대상 : 만 65세 이상,의료비 지원과 교육·알림상담 대상 연령이 다름. JA 65세 이상만으로 30~64세...
1,542000000117,age,ja_text_difference,만 65세 이상 농업인,JA 시작 70과 본문 시작 65가 다름. 수집시점 원문상 65~69세가 JA에서 ...
2,999000000036,age,multiple_priority_groups,5∼69세 장애인,전체 연령 5~69와 우선순위별 5~18/19세 이상을 구분. 텍스트 첫 매칭 19...
3,999000000021,age,multiple_programs,무주택자 청년(19세~39세),일반·청년·신혼부부 여러 유형을 하나의 연령 구간으로 판정하지 않음. 자녀 나이도 ...
4,453000000204,age,household_member_age,만 18세 미만 손자녀,손자녀 연령이 신청자 전체 상한이 아님. 노인·장애인·한부모 등 세대 조건을 분리.
5,460000000162,age,different_steps,상품권 구매 : 만19세이상,구매와 설치·발급 단계별 하한이 다름. JA 14세 이상을 할인구매 자격으로 쓰지 않음.
6,O00005600002,age,disjoint_groups_and_exclusive_bound,만 19세 미만,노인·청소년·아동·다른 대상군의 합집합. 텍스트 min65/max19를 연속구간으로...
7,O00027200002,age,child_age_and_multiple_groups,막내가 만 15세 이하,막내 자녀 15세 조건은 신청자 상한이 아님. 수영장 여성 등 다른 혜택 조건도 분리.
8,B27000100042,income,ja_band_not_exact_cap,기준중위소득 150% 이하,JA 상한 100과 본문 150. JA 구간 경계 표시는 실제 100% 상한을 확정...
9,B27000100016,income,ja_band_not_exact_cap,기준중위소득 125% 이하,JA 상한 100과 본문 125. JA만으로 100~125% 대상자를 배제하지 않음.


**한계:** 현재 규칙 기준 개인 미확인 건수는 기존 163건에서 조건부 자동신청 2건이 해소되어 161건이다. 별도로 신청불필요 오분류 1건이 미확인으로 바뀌었으나 비개인 대상이라 이 분모에 포함되지 않는다. AI 38건은 기존 163건에 대한 별도 검토 수이며 규칙 미확인 건수에서 일괄 차감하지 않는다.

## PART D. 출력·재현성 검증

새 회귀 테스트는 웹사이트/실방문 3사례, 별도 신청/불필요 부정문, 문의 오인, 출처·근거 보존, 무손실 병합 및 키 오류 중단을 점검한다. 전체 기존 테스트도 실행했으나 `test_supplement_sample_is_independent` 1건은 Git 제외된 선행 `outputs/02_labeling/tables/labeling_key.csv`가 이 작업트리에 없어 실패했다(나머지 141건 통과). 보조금24 작업으로 기존 라벨링 정답 시트를 생성·교체하지 않았다.

화면 검증은 컴퓨터 사용 도구의 브라우저 URL 확인 실패로 중단됐다. 아래 HTML 검사는 오프라인 자원 구조·앵커·표 존재에 대한 정적 검사이며 실제 넓은/좁은 화면·키보드·인쇄 검증 통과를 뜻하지 않는다.

In [6]:
test=subprocess.run([sys.executable,"-m","pytest","tests/test_gov24_preparation.py","-q"],cwd=ROOT,capture_output=True,text=True,encoding="utf-8")
print(test.stdout)
assert test.returncode==0,test.stderr
from html.parser import HTMLParser
class ReportCheck(HTMLParser):
    def __init__(self):
        super().__init__(); self.ids=[]; self.anchors=[]; self.externals=[]; self.tables=0
    def handle_starttag(self,tag,attrs):
        a=dict(attrs)
        if "id" in a: self.ids.append(a["id"])
        if tag=="a" and a.get("href","").startswith("#"): self.anchors.append(a["href"][1:])
        if (tag in ["script","img","iframe"] and a.get("src") and not a["src"].startswith("data:")) or (tag=="link" and a.get("rel")=="stylesheet"): self.externals.append(a)
        if tag=="table": self.tables+=1
report=ROOT/"outputs"/STAGE/"summary"/f"{STAGE}_summary_final.html"
check=ReportCheck(); check.feed(report.read_text(encoding="utf-8"))
assert len(check.ids)==len(set(check.ids)) and set(check.anchors)<=set(check.ids)
assert not check.externals and check.tables==6
for rel,expected in manifest["outputs_sha256"].items(): assert sha(ROOT/rel)==expected,rel
for rel,expected in manifest["protected_sha256"].items(): assert sha(ROOT/rel)==expected,rel
print({"html_tables":check.tables,"toc_anchors":len(check.anchors),"external_assets":len(check.externals),"ui_rendering":"미검증"})

..............                                                           [100%]
14 passed in 0.15s

{'html_tables': 6, 'toc_anchors': 5, 'external_assets': 0, 'ui_rendering': '미검증'}


## 최종 인수 기록 · 전체 완료 아님

| 번호 | 항목 | 상태 | 근거 |
|---|---|---|---|
| 1 | 범위·입출력 | 통과 | 실제 파일 목록·보존 경로·로그 |
| 2 | 노트북 형식·환경 | 통과 | JSON·Python·pandas·기준 커밋·해시·난수 기록 |
| 3 | 실행·출력 보존 | 통과(대체 실행) | 새 Python 프로세스에서 위→아래 순차 실행·실제 출력 저장. Jupyter 커널 시작은 미검증 |
| 4 | 정의·결정 | 통과 | AI/사람·원문/후보·기존/v2·줄바꿈/규칙 차이 분리 |
| 5 | 결과표 검증 | 통과 | 행·키·문자열·근거·해시 대조 |
| 6 | 결과그림 | 해당 없음 | 필드·사례 검증은 표로 보고 |
| 7 | 매체 정합성 | 통과 | 동일 계산의 CSV를 읽어 HTML·노트북 표시, 해시 대조 |
| 8 | HTML 독립성·탐색 | 미검증 | 정적 앵커/외부 자원 검사 통과, 실제 브라우저 오프라인 미검증 |
| 9 | 화면·접근성·인쇄 | 미검증 | 컴퓨터 사용 도구가 브라우저 URL을 확인하지 못해 중단 |
| 10 | 해석·공유 | 통과 | 미확인 125건·AI 잠정 판정·과거 상세 13항목 미검증 공개 |
| 11 | 최종 기록 | 통과(부분 완료 기록) | 산출물·한계·재현 절차·보호 해시 보존 |

**남은 검증:** 수집 당시 상세 13개 필드 원본 확보, 실제 브라우저 화면 검증, 사람 2인 독립 판정·합의 및 정답 기반 성능 평가. Q2 연결 후보 ID별 포함 관계는 대용량 파일 미반환으로 미검증이며 메타데이터의 입력 해시·건수만 확인했다.

**재현:** 저장소 루트에서 `python -m src.preprocess.gov24_preparation --refresh` 후 이 노트북을 초기 커널에서 순차 실행한다. `--refresh`는 이 보충 단계 생성물만 갱신하며 기존 `gov24_*.csv`·Q2·SSIS 및 사람이 기입한 검수 시트는 보존한다. 생성 결과물·AI 판정 로그를 자동 커밋하거나 푸시하지 않았다.